# Variant clusters

Qualifying disease credible sets are grouped into likely independent causal signals:
connected components over significant colocalisations and shared lead variants. The number
of distinct diseases in a cluster is its variant pleiotropy score (vPS). Methods
"Variant-level pleiotropy modelling".

Disease identity uses the ontology-resolved `diseaseIds` column, as everywhere else in the
pipeline. The raw curator column `traitFromSourceMappedIds` is reported alongside for
comparison: 26 of its ids no longer exist in the release ontology.

Writes `variant_clusters`, `cluster_membership`.

In [ ]:
import pandas as pd

from manuscript_methods import clusters, paper

In [ ]:
cs = clusters.load_credible_sets()
print("qualifying disease credible sets:", len(cs))

edges = clusters.load_edges(set(cs["studyLocusId"]))
print("colocalisation edges within the set:", len(edges))

components = clusters.cluster(list(zip(cs["studyLocusId"], cs["variantId"])), edges)
print("clusters:", len(components))
print("credible sets assigned:", sum(len(m) for _, m in components))

## Per-cluster counts

In [ ]:
table = clusters.cluster_table(cs, components)
table.to_parquet(paper.derived("variant_clusters"), index=False)

print("clusters with more than one lead variant:", int((table["uniqueLeadVariants"] > 1).sum()))
print("clusters with more than one disease:", int((table["uniqueDiseases"] > 1).sum()))
print("clusters with more than one therapeutic area:", int((table["uniqueTherapeuticAreas"] > 1).sum()))
print(table[["uniqueDiseases", "uniqueTherapeuticAreas"]].agg(["min", "max", "mean"]).round(4).to_string())

## Cluster membership, for Supplementary Table 15

In [ ]:
membership = clusters.membership_table(cs, components)
membership.to_parquet(paper.derived("cluster_membership"), index=False)
print("cluster-disease rows:", len(membership))
membership.head(3)

## Control: the raw curator column reproduces the originally published counts

In [ ]:
raw = clusters.cluster_table(cs, components, trait_column="traitFromSourceMappedIds")
comparison = pd.DataFrame(
    {
        "raw (originally published)": [
            (raw["uniqueDiseases"] > 1).sum(),
            raw["uniqueDiseases"].max(),
            raw["uniqueDiseases"].mean(),
            (raw["uniqueTherapeuticAreas"] > 1).sum(),
            raw["uniqueTherapeuticAreas"].max(),
            raw["uniqueTherapeuticAreas"].mean(),
        ],
        "resolved (used here)": [
            (table["uniqueDiseases"] > 1).sum(),
            table["uniqueDiseases"].max(),
            table["uniqueDiseases"].mean(),
            (table["uniqueTherapeuticAreas"] > 1).sum(),
            table["uniqueTherapeuticAreas"].max(),
            table["uniqueTherapeuticAreas"].mean(),
        ],
    },
    index=[
        "clusters with >1 disease",
        "max diseases",
        "mean diseases",
        "clusters with >1 therapeutic area",
        "max therapeutic areas",
        "mean therapeutic areas",
    ],
).round(4)
comparison